# Mart: IP charges (annual time-grain) fact table

Reads the staged OECD IP charges data and writes the final dashboard-ready fact table.

| | |
|---|---|
| **Reads** | `data/staging/oecd/stg_oecd__ip_charges.csv` |
| **Writes** | `data/marts/fct_ip_charges_yearly.csv` |

**Output schema:**

| Column | Type | Description |
|---|---|---|
| `year` | int | Reference year |
| `counterpart_country` | str | Trading partner full name (e.g. `"United States"`) |
| `flow_type` | str | Always `"Charges for use of IP - Balance"` |
| `value_cad_millions` | float | Net value in millions CAD (negative = Canada paid more than received) |

In [ ]:
import pandas as pd
from pathlib import Path
import sys

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from utils.paths import STG_OECD_DIR, MARTS_DIR

MARTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
INPUT_FILENAME  = 'stg_oecd__ip_charges.csv'
OUTPUT_FILENAME = 'fct_ip_charges_yearly.csv'

FLOW_TYPE = 'Charges for use of IP - Balance'

df = pd.read_csv(STG_OECD_DIR / INPUT_FILENAME)
print(f'Loaded {len(df)} rows from staging')
df.head()

Loaded 1315 rows from staging


,year,counterpart_area,counterpart_country,value_cad_millions
0,1969,W,World,-167.0
1,1970,W,World,0.0
2,1971,W,World,0.0
3,1972,W,World,0.0
4,1973,W,World,-243.0


In [ ]:
# Align OECD country names to Dim MacroRegion naming convention in Power BI.
# Mapping uses ISO area codes (from counterpart_area) to avoid special character issues.
AREA_CODE_NAME_MAP = {
    'CHN': 'China',
    'HKG': 'Hong Kong',
    'SVK': 'Slovakia',
    'KOR': 'South Korea',
    'TWN': 'Taiwan',
}

fct_df = (
    df
    .assign(flow_type=FLOW_TYPE)
    .assign(counterpart_country=lambda x: x['counterpart_area'].map(AREA_CODE_NAME_MAP).fillna(x['counterpart_country']))
    [['year', 'counterpart_country', 'flow_type', 'value_cad_millions']]
    .sort_values(['year', 'counterpart_country'])
    .reset_index(drop=True)
)

print(f'Final mart shape: {fct_df.shape}')
print(f'Countries: {sorted(fct_df["counterpart_country"].unique().tolist())}')
fct_df.head(10)

In [ ]:
output_path = MARTS_DIR / OUTPUT_FILENAME
fct_df.to_csv(output_path, index=False, encoding='utf-8')
print(f'Fact table saved to: {output_path}')

Fact table saved to: C:\Users\GuidaK\OneDrive - ISED-ISDE\Documents\Work\or_country_profiles_dashboard\data\marts\fct_ip_charges_yearly.csv
